# Анализ данных на Python
## Операции с датафреймами Pandas: часть 2

*Алла Тамбовцева, НИУ ВШЭ*

Содержание конспекта:

* Обработка текста: метод `.str.extract()` и регулярные выражения
* Обработка текста: метод `str.split()` и разделение текста на столбцы
* Обработка даты и формат `datetime` в `pandas`

Кроме того:

* Выбор столбцов списком, методы  `.loc` и `.iloc`
* Пример заполнения пропусков
* Объединение датафреймов через функцию `.concat()`

Импортируем библиотеку `pandas` и выключим сообщения с предупреждениями вида `chained_assignment`. Такие сообщения возникают, когда мы создаём новые столбцы на основе старых с помощью методов `pandas`, но при этом выбираем их просто по названию, а не через метод `.loc`. Пока это некритично, во избежание более громоздкого кода сдадимся и уберрём предупреждения.

In [1]:
import pandas as pd
pd.set_option('chained_assignment', None)

Загрузим данные из файла `hseteachers.csv`, в нём сохранены результаты выгрузки постов сообщества «Цитатник ВШЭ» ВКонтакте через API. 

> Про получение доступа к API ВКонтакте и пример выгрузки постов (на примере другого сообщества) – см. дополнительные материалы.

In [2]:
posts = pd.read_csv("https://raw.githubusercontent.com/allatambov/PyPerm25/refs/heads/main/hseteachers.csv")

Выберем основные столбцы:
    
* `id`: id поста;
* `date`: дата публикации поста в формате UNIX;
* `text`: текст поста;
* `likes`: характеристики лайков к посту;
* `reposts`: характеристики репостов к посту;
* `views`: характеристики просмотров к посту;
* `comments`: характеристики комментариев к посту.

In [3]:
# выбор списком
small = posts[["id", "date", "text", "likes", 
               "reposts", "views", "comments"]]

Посмотрим на первые несколько строк:

In [4]:
print(small.head(2))

      id        date                                               text  \
0  40491  1744982557  Если вы общаетесь с Богом - это хорошо,\nА вот...   
1  40489  1744900727  Студенты: на лекции про революцию Мэйдзи в Япо...   

                                               likes  \
0  {'can_like': 1, 'count': 20, 'user_likes': 0, ...   
1  {'can_like': 1, 'count': 30, 'user_likes': 0, ...   

                            reposts            views  \
0  {'count': 8, 'user_reposted': 0}  {'count': 1003}   
1  {'count': 2, 'user_reposted': 0}  {'count': 1888}   

                                     comments  
0  {'can_post': 1, 'can_view': 1, 'count': 1}  
1  {'can_post': 1, 'can_view': 1, 'count': 0}  


Изначально при выгрузке данных через API результат был получен в виде огромной JSON-строки, которая в Python автоматически десериализовались в словарь. Из этого словаря был извлечён список словарей с постами (один пост = один словарь с большим числом записей), который был превращён в датафрейм.

Если бы мы обрабатывали такой датафрейм сразу после выгрузки, число лайков, репостов, комментариев можно было бы извлечь по ключу `'count'` из словаря в каждой ячейке соответствующего столбца. Примерно так:

       small["likes"].apply(lambda x: x['count'])

Однако здесь мы имеем дело не с оригинальным датафреймом, а с его версией после выгрузки и считывания из текстового CSV-файла. Это усложняет задачу – в ячейках уже не словари, а строки с разметкой, как у словаря! Логичное желание – десериализовать их как JSON-строки. Но, если внимательно посмотреть, записи вида `{'can_post': 1, 'can_view': 1, 'count': 0}` валидными JSON-строками не являются. 

Можно, конечно, превратить через умную замену с регулярными выражениями сделать эти записи валидными JSON-строками (кавычки «накинуть») и вернуться к вопросу десериализации. Но мы поступим по-другому – воспользуемся теми же регулярными выражениями и извлечём нужные числа через метод `.str.extract()`.

Метод `str.extract()` позволяет извлечь группы символов в рамках регулярного выражения. А значит, достаточно написать шаблон, указывающий на то, что после `'count': ` стоит группа цифр, которую мы хотим забрать:

In [5]:
pattern = "'count':\s(\d+)"

small["likes"].str.extract(pattern)

,0
0,20
1,30
2,24
3,1
4,32
...,...
10156,6
10157,19
10158,40
10159,24


Работает! К слову, если групп было бы несколько в рамках одного выражения, каждая группа была бы сохранена в отдельном столбце, что удобно.

Осталось проделать эту операцию для всех однотипных столбцов и привести тип столбцов к целочисленному (сейчас всё ещё `object`):

In [6]:
small["nlikes"] = small["likes"].str.extract(pattern).astype(int)
small["nreposts"] = small["reposts"].str.extract(pattern).astype(int)
small["ncomments"] = small["comments"].str.extract(pattern).astype(int)

Со столбцом `views` всё чуть-чуть иначе. Так как у старых постов просмотры не фиксировались, в этом столбце встречаются пропуски. А если в столбце с целыми числами есть хотя бы один `NaN`, он автоматически становится `float`. Поэтому превращение в `int` здесь не получится. 

Преобразуем тип в `float`, а следующим шагом заполним пропуски нулями (здесь допустимо, посты с 0 просмотрами отсутствуют). А потом уже превратим все к типу `int`:

In [7]:
# fillna() – заполнение пропусков
# здесь – фиксированным значением

small["nviews"] = small["views"].str.extract(pattern).astype(float)
small["nviews"] = small["nviews"].fillna(0).astype(int)

Удалим старые столбцы, из которых мы уже извлекли всё необходимое:

In [8]:
small.drop(columns = ["likes", "reposts", "comments", "views"], 
           inplace = True)

Разберёмся с датой. Сейчас в столбце `date` сохранены дата-время в формате POSIX, число секунд с 1 января 1970 года. Чтобы получить что-то более наглядное, приведём столбец к формату `datetime` в `pandas`:

In [9]:
# unit = 's', так как переводим секунды

small["date"] = pd.to_datetime(small["date"], unit = "s")
print(small.head(2))

      id                date  \
0  40491 2025-04-18 13:22:37   
1  40489 2025-04-17 14:38:47   

                                                text  nlikes  nreposts  \
0  Если вы общаетесь с Богом - это хорошо,\nА вот...      20         8   
1  Студенты: на лекции про революцию Мэйдзи в Япо...      30         2   

   ncomments  nviews  
0          1    1003  
1          0    1888  


Тип `datetime` удобен тем, что данные такого типа можно полноценно сортировать в соответствии с хронологией, отображать на графиках для визуализации динамики, и прочее. Но ещё его можно превратить в строку с заданным форматированием. А из этой строки уже извлечь необходимый фрагмент даты-времени – год, день недели или часы-минуты-секунды:

In [10]:
# забираем части даты-времени в виде строки (тип string)
# %Y – год в четырехзначном виде, %y – год в двузначном виде
# %m – месяц в числовом виде
# %d – день в числовом виде
# %A – день недели полностью, %a – день недели сокращенно
# %H, %M, %S – часы, минуты, секунды

small["year"] = small["date"].dt.strftime("%Y")
small["month"] = small["date"].dt.strftime('%m')
small["day"] = small["date"].dt.strftime('%d')
small["wday"] = small["date"].dt.strftime("%A")
small["time"] = small["date"].dt.strftime('%H:%M:%S')

print(small.head(2))

      id                date  \
0  40491 2025-04-18 13:22:37   
1  40489 2025-04-17 14:38:47   

                                                text  nlikes  nreposts  \
0  Если вы общаетесь с Богом - это хорошо,\nА вот...      20         8   
1  Студенты: на лекции про революцию Мэйдзи в Япо...      30         2   

   ncomments  nviews  year month day      wday      time  
0          1    1003  2025    04  18    Friday  13:22:37  
1          0    1888  2025    04  17  Thursday  14:38:47  


Осталось только разобраться с текстом поста. Посмотрим на пример одного поста:

In [11]:
print(small["text"][0])

Если вы общаетесь с Богом - это хорошо,
А вот если Бог начинает общаться с вами - это уже проблема.

#ВШЭСПБ #Юрфак #Закревский_ВШЭ


Здесь явно есть сама цитата преподавателя, а затем – тэги. Можем разделить текст на части по `#`:

In [12]:
small["text"].str.split("#")

0        [Если вы общаетесь с Богом - это хорошо,\nА во...
1        [Студенты: на лекции про революцию Мэйдзи в Яп...
2        [«С точки зрения Сергея Караганова: ядерный уд...
3        [Навалилось много дел по учебе, ничего не успе...
4        [Вообще, расцвет математики и матанализа прише...
                               ...                        
10156    [по маленькой кругленькой громов\n, Громов_hse...
10157       [Все глоки суть куздры (Данько)\n, Данько_hse]
10158    [с какого бадуна ты это написал? (Самовол)\n, ...
10159    [Синдром яндекса (Шаповалов И. А)\n, Шаповалов...
10160    [Задача тривиальна \n(Акимов Д.В.)\n, Акимов_hse]
Name: text, Length: 10161, dtype: object

Проблема: после такого разделения в столбце окажутся ячейки со списками. Сейчас это не такая большая проблема, но при выгрузке в тот же CSV или Excel, эти списки станут неудобными строками с запятыми и квадратными скобками (как словари в самом начале, которые пришлось обрабатывать через `str.extract()`). 

Решение: добавим аргумент `expand = True`, он растянет элементы списков на отдельные столбцы:

In [13]:
new = small["text"].str.split("#", expand = True)
new.head(2)

,0,1,2,3,4,5,6
0,"Если вы общаетесь с Богом - это хорошо,\nА вот...",ВШЭСПБ,Юрфак,Закревский_ВШЭ,None,None,None
1,Студенты: на лекции про революцию Мэйдзи в Япо...,Суздальцев_ВШЭ,ПЭИ,None,None,None,None


В `new` сохранён новый датафрейм из новых «разбитых» столбцов. Выберем первые три столбца, используя метод `.iloc` или `.loc`.

> В `pandas` на датафреймах определены два метода:
  * метод `.iloc` для выбора строк/столбцов по индексам (*index location*);
  * метод `.loc` для выбора строк/столбцов по названиям (*location*), но по индексам тоже выбор возможен.

In [14]:
# iloc – срез 0:3, правый конец 3 не включается (0, 1, 2)
# loc – срез 0:2, правый конец включается (0, 1, 2)
# оба варианта одинаковы

add = new.iloc[:, 0:3]
add = new.loc[:, 0:2]

Припишем названия выбранным столбцам – глобальный список для всего датафрейма:

In [15]:
add.columns = ["phrase", "name", "place"]

Осталось объединить новый датафрейм `add` с основным `small`. Воспользуемся функцией `.concat()`. По умолчанию она объединяет датафреймы по строкам, для объединения по столбцам добавим аргумент `axis = 1` (ось `axis = 0` – строки, `axis = 1` – столбцы):

In [16]:
final = pd.concat([small, add], axis = 1)
final.head(2)

,id,date,text,nlikes,nreposts,ncomments,nviews,year,month,day,wday,time,phrase,name,place
0,40491,2025-04-18 13:22:37,"Если вы общаетесь с Богом - это хорошо,\nА вот...",20,8,1,1003,2025,04,18,Friday,13:22:37,"Если вы общаетесь с Богом - это хорошо,\nА вот...",ВШЭСПБ,Юрфак
1,40489,2025-04-17 14:38:47,Студенты: на лекции про революцию Мэйдзи в Япо...,30,2,0,1888,2025,04,17,Thursday,14:38:47,Студенты: на лекции про революцию Мэйдзи в Япо...,Суздальцев_ВШЭ,ПЭИ


Можем выгрузить результат в CSV:

In [17]:
final.to_csv("hse_upd.csv")